In [79]:
import torch
from torch import nn

class SmallNetwork(nn.Module):
    def __init__(self, l2i, l3i, l4i):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3, l2i),
            nn.ReLU(),
            nn.Linear(l2i, l3i),
            nn.ReLU(),
            nn.Linear(l3i, l4i),
            nn.ReLU(),
            nn.Linear(l4i, 1),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

model = SmallNetwork(3,3,3)

# inspect weights
for name, param in model.named_parameters():
    print(name, param.shape, '\n', param.data)


linear_relu_stack.0.weight torch.Size([3, 3]) 
 tensor([[ 0.1865,  0.3033,  0.2868],
        [ 0.5694,  0.2278, -0.4184],
        [ 0.1973,  0.1671, -0.5602]])
linear_relu_stack.0.bias torch.Size([3]) 
 tensor([-0.1692, -0.1592,  0.1222])
linear_relu_stack.2.weight torch.Size([3, 3]) 
 tensor([[ 0.2304,  0.2407, -0.4381],
        [-0.2657,  0.1979, -0.4959],
        [-0.4615, -0.4696,  0.3183]])
linear_relu_stack.2.bias torch.Size([3]) 
 tensor([ 0.3535, -0.4193, -0.3957])
linear_relu_stack.4.weight torch.Size([3, 3]) 
 tensor([[ 0.2134,  0.3418,  0.5743],
        [ 0.2537, -0.2982, -0.2635],
        [-0.1872, -0.5250, -0.0255]])
linear_relu_stack.4.bias torch.Size([3]) 
 tensor([ 0.5056, -0.4737, -0.1709])
linear_relu_stack.6.weight torch.Size([1, 3]) 
 tensor([[-0.5032, -0.5333, -0.2221]])
linear_relu_stack.6.bias torch.Size([1]) 
 tensor([-0.3179])


In [80]:
linear_layers = [(i, m) for i, m in enumerate(model.linear_relu_stack) if isinstance(m, nn.Linear)]

saved = [[layer.weight.data.clone(), layer.bias.data.clone()] 
         for _, layer in linear_layers]

In [81]:
len(saved)

4

In [82]:
def remove(saved, layer, node):
    # remove row from current layer weight
    saved[layer-1][0] = torch.cat([saved[layer-1][0][:node], saved[layer-1][0][node+1:]], dim=0)
    # remove element from current layer bias
    saved[layer-1][1] = torch.cat([saved[layer-1][1][:node], saved[layer-1][1][node+1:]])
    # remove column from next layer weight
    saved[layer][0] = torch.cat([saved[layer][0][:, :node], saved[layer][0][:, node+1:]], dim=1)

def leastweight(saved):
    layer = 0
    node = 0
    v = torch.inf
    for l in range(1,len(saved)):
        importance = saved[l][0].abs().sum(dim=0)
        print(importance)
        if v >= min(importance).item():
            v = min(importance).item()
            layer = l
            node = importance.argmin().item()
    return layer, node

In [83]:
print(saved[0][0].shape[0], saved[1][0].shape[0], saved[2][0].shape[0])

3 3 3


In [84]:
leastweight(saved)


tensor([0.9576, 0.9081, 1.2523])
tensor([0.6543, 1.1651, 0.8633])
tensor([0.5032, 0.5333, 0.2221])


(3, 2)

In [85]:
remove(saved, 3, 2)

In [86]:
print(saved[0][0].shape[0], saved[1][0].shape[0], saved[2][0].shape[0])

3 3 2


In [87]:
leastweight(saved)

tensor([0.9576, 0.9081, 1.2523])
tensor([0.4671, 0.6400, 0.8378])
tensor([0.5032, 0.5333])


(2, 0)

In [88]:
l2i = saved[1][0].shape[0]  # out_features of layer 0
l3i = saved[2][0].shape[0]  # out_features of layer 1
l4i = saved[3][0].shape[0]  # out_features of layer 2

new_model = SmallNetwork(l2i, l3i, l4i)
new_linear_layers = [(i, m) for i, m in enumerate(new_model.linear_relu_stack) if isinstance(m, nn.Linear)]

for (w, b), (_, layer) in zip(saved, new_linear_layers):
    layer.weight.data = w.clone()
    layer.bias.data   = b.clone()

In [89]:
for name, param in new_model.named_parameters():
    print(name, param.shape, '\n', param.data)

linear_relu_stack.0.weight torch.Size([3, 3]) 
 tensor([[ 0.1865,  0.3033,  0.2868],
        [ 0.5694,  0.2278, -0.4184],
        [ 0.1973,  0.1671, -0.5602]])
linear_relu_stack.0.bias torch.Size([3]) 
 tensor([-0.1692, -0.1592,  0.1222])
linear_relu_stack.2.weight torch.Size([3, 3]) 
 tensor([[ 0.2304,  0.2407, -0.4381],
        [-0.2657,  0.1979, -0.4959],
        [-0.4615, -0.4696,  0.3183]])
linear_relu_stack.2.bias torch.Size([3]) 
 tensor([ 0.3535, -0.4193, -0.3957])
linear_relu_stack.4.weight torch.Size([2, 3]) 
 tensor([[ 0.2134,  0.3418,  0.5743],
        [ 0.2537, -0.2982, -0.2635]])
linear_relu_stack.4.bias torch.Size([2]) 
 tensor([ 0.5056, -0.4737])
linear_relu_stack.6.weight torch.Size([1, 2]) 
 tensor([[-0.5032, -0.5333]])
linear_relu_stack.6.bias torch.Size([1]) 
 tensor([-0.3179])


In [92]:
ll = [(i, m) for i, m in enumerate(new_model.linear_relu_stack) if isinstance(m, nn.Linear)]

saved2 = [[layer.weight.data.clone(), layer.bias.data.clone()] 
         for _, layer in ll]

leastweight(saved2)

tensor([0.9576, 0.9081, 1.2523])
tensor([0.4671, 0.6400, 0.8378])
tensor([0.5032, 0.5333])


(2, 0)